# 03 LoRA 训练 Colab

本 notebook 当前只执行 `05A LoRA 训练前探测`：加载 `Qwen/Qwen3-ASR-1.7B`，导出真实模块结构，并生成第一版 LoRA target 候选。完成探测并复核 target 后，再扩展为 5-20 step smoke training。

In [ ]:
# 挂载 Google Drive。
# 项目目录默认使用 /content/drive/MyDrive/qwen3-asr。
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# 安装最小依赖。
# only-if-needed 可以降低 Colab 预装包被过度升级的风险。
# 不要安装未固定版本的 pandas；Colab 当前依赖 pandas==2.2.2。
%pip -q install --upgrade --upgrade-strategy only-if-needed qwen-asr huggingface_hub pyyaml
%pip -q install pandas==2.2.2


In [ ]:
# 读取项目路径和训练配置。
# 如果你的 Drive 目录名不同，只需要改 PROJECT_DIR。
from pathlib import Path
import yaml

PROJECT_DIR = Path('/content/drive/MyDrive/qwen3-asr')
CONFIG_PATH = PROJECT_DIR / 'configs/train/qwen3_asr_lora_mvp.yaml'

assert PROJECT_DIR.exists(), f'项目目录不存在: {PROJECT_DIR}'
assert CONFIG_PATH.exists(), f'训练配置不存在: {CONFIG_PATH}'

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

def project_path(value: str) -> Path:
    """把配置中的相对路径解析到 PROJECT_DIR 下，绝对路径保持不变。"""
    path = Path(value)
    return path if path.is_absolute() else PROJECT_DIR / path

MODEL_ID = config['model']['id']
PROBE = config.get('probe', {})
OUTPUT_DIR = project_path(PROBE.get('output_dir', 'outputs/lora_probe/qwen3_asr_1_7b'))
DTYPE = PROBE.get('dtype', 'float16')
DEVICE_MAP = PROBE.get('device_map', 'cuda:0')
MAX_INFERENCE_BATCH_SIZE = int(PROBE.get('max_inference_batch_size', 1))
MAX_NEW_TOKENS = int(PROBE.get('max_new_tokens', 128))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('PROJECT_DIR =', PROJECT_DIR)
print('MODEL_ID =', MODEL_ID)
print('OUTPUT_DIR =', OUTPUT_DIR)


In [ ]:
# 检查 GPU。
# 如果这里没有 CUDA，先在 Colab 菜单中选择 Runtime -> Change runtime type -> GPU。
import torch

print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))


## Hugging Face 登录

如果模型下载遇到权限或限流问题，先执行下面单元登录 Hugging Face。已经登录或模型可直接下载时，可以跳过。

In [ ]:
# 可选：登录 Hugging Face。
# from huggingface_hub import notebook_login
# notebook_login()


In [ ]:
# 执行 Qwen3-ASR 模块探测。
# 输出会写入 OUTPUT_DIR，后续可以直接提交这些文件用于排查和复核。
import subprocess
import sys

cmd = [
    sys.executable,
    'train/inspect_qwen3_asr_modules.py',
    '--model-id', MODEL_ID,
    '--output-dir', str(OUTPUT_DIR),
    '--dtype', DTYPE,
    '--device-map', DEVICE_MAP,
    '--max-inference-batch-size', str(MAX_INFERENCE_BATCH_SIZE),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
]

print('运行命令:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_DIR), check=True)


In [ ]:
# 预览探测输出。
# 重点查看候选分组数量、leaf module names，以及是否存在 speech/audio 相关候选。
import json

for path in sorted(OUTPUT_DIR.glob('*')):
    print(path.name, path.stat().st_size, 'bytes')

candidates_path = OUTPUT_DIR / 'lora_target_candidates.json'
assert candidates_path.exists(), f'缺少候选 target 文件: {candidates_path}'

payload = json.loads(candidates_path.read_text(encoding='utf-8'))
print(json.dumps(payload.get('groups', {}), ensure_ascii=False, indent=2)[:4000])


## 完成标准

本阶段跑通后，请确认 `outputs/lora_probe/qwen3_asr_1_7b/` 中至少包含：

- `module_snapshot.json`
- `module_summary.csv`
- `lora_target_candidates.json`
- `lora_target_candidates.md`

下一步会根据这些输出决定第一版 LoRA smoke training 的 target 组。